In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from eucare.base import *
import eucare.plotting as euplot


plt.figure(figsize=(5, 5))
prev_poly = regular_poly_points(3)
for i in range(3, 20):
    poly = regular_poly_points(i) * np.random.rand()
    j = np.random.randint(0, i-2)
    mat = find_affine(poly[j:j+2][::-1], prev_poly[i-3:i-1])
    poly = apply_affine(poly, mat)
    euplot.plot_polygon(poly)
    prev_poly=poly
    #plt.scatter(*(poly[j:j+2].T))
    
euplot.set_equal_aspect()

## HalfEdge Data Structure

In [ ]:
from eucare.half import AttributeObject

a = AttributeObject()

print(a.has_attributes())
a['adsf'] = 'party'
print(a.has_attributes())
for key, val in a.items():
    print(key, val)


In [ ]:
from eucare.half import Vertex, CyclicHalfedgeGraph, IdObject
from tqdm import tqdm_notebook as tqdm
import networkx as nx

IdObject.reset_ids()
poly = CyclicHalfedgeGraph([Vertex() for i in range(4)])
#for v in poly.vertices:
#    print(v)
#    for h in v.outgoing_iter():
#        pass
#        print(h.orig, h.dest)

#hs = list(poly.halfedges)

#f = any_element(poly.faces)
#print(f.__dict__)
#[print(v) for v in f.reverse_halfedge_iter()]


for i in range(3):
    print(i, poly.order)
    border_vertices = poly.border_vertices()
    for h1 in tqdm(list(poly.border_edge_iter())):
        #print(h1)
        to_attach = CyclicHalfedgeGraph([Vertex() for i in range(5)])
        h2 = to_attach.get_any_border()
        poly.add_graph(to_attach)
        poly.glue_e2e(h1, h2)
    for v in border_vertices:
        poly.close_vertex(v)
        

G = poly.to_networkx_undirected()
pos = nx.spring_layout(G.to_undirected())
plt.figure(figsize=(15, 15))
nx.draw_networkx_nodes(G, pos, cmap=plt.get_cmap('jet'), node_size = 500)
nx.draw_networkx_edges(G, pos, edge_color='r', arrows=True)
nx.draw_networkx_labels(G, pos);

print(poly.get_any_border())

In [ ]:
import numpy as np
import collections

class EuclideanVertex2D(Vertex):
    def __init__(self, pos, any_outgoing=None):
        super(EuclideanVertex2D, self).__init__(any_outgoing)
        if not isinstance(pos, colledctions.Sized):
            raise ValueError(f"position must be Sized. Got {pos}.")
        if len(pos) != 2:
            raise ValueError(f"Got position of length {len(pos)} != 2.")
        self.pos = np.array(pos, dtype=np.float32)
    
    @property
    def x(self):
        return self.pos[0]
    
    @property
    def y(self):
        return self.pos[1]

In [ ]:
v1, v2 = Vertex(), Vertex()
h1, h2 = HalfEdge(orig=v1, dest=v2), HalfEdge(orig=v2, dest=v1)
h1.rev = h2
h2.rev = h1
e = Edge(h1, h2)

In [ ]:
e[h1.orig]

In [ ]:
from copy import copy

v = Vertex()
d = dict()
d[v] = 3
v.any_outgoing = 123
d[v]